# ACLED Data Pipeline: Databricks -> Unity Catalog -> ArcGIS Online

This notebook reads the ACLED Armed Conflict Location and Event Dataset from the FCV team's SQL Server database, optionally writes raw and cleaned Delta tables for discovery in Databricks, and publishes selected global, regional, or country layers to ArcGIS Online.

| Property | Value |
|---|---|
| Source | `dbo.acled_country_data` - Azure SQL Server via JDBC |
| Databricks outputs | Configurable Delta tables, defaulting to `hive_metastore.acled.acled_raw` and `hive_metastore.acled.acled_clean` |
| AGOL portal | `https://geowb.maps.arcgis.com` by default |
| AGOL auth | Databricks secrets, scope `agol-credentials`, keys `username` and `password` |
| AGOL metadata | `src/mdp/data/acled_metadata.yml` |
| Default safety | `dry_run=true`, `publish_to_agol=false`, `write_to_unity_catalog=false`, `source_mode=auto` |

## Run order
1. Set widgets at the top of the notebook. The default run checks whether the SQL source changed and reuses the raw Delta cache when possible.
2. For Databricks table writes, set `write_to_unity_catalog=true` and `dry_run=false`.
3. For AGOL publishing, set `publish_to_agol=true` and `dry_run=false` after status checks pass.
4. If a new AGOL item is created, copy the printed item ID into `src/mdp/data/acled_metadata.yml`.


In [ ]:
# Imports and Databricks widget helpers
import json
import math
import os
import re
from datetime import date
from pathlib import Path

import requests
import yaml
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    ByteType,
    DateType,
    DecimalType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    ShortType,
    StringType,
    TimestampType,
)

spark = SparkSession.builder.getOrCreate()

try:
    dbutils
    HAS_DBUTILS = True
except NameError:
    HAS_DBUTILS = False


def _ensure_text_widget(name, default, label=None):
    if HAS_DBUTILS:
        try:
            dbutils.widgets.get(name)
        except Exception:
            dbutils.widgets.text(name, default, label or name)
        return dbutils.widgets.get(name)
    return os.getenv(name.upper(), default)


def _ensure_dropdown_widget(name, default, choices, label=None):
    if HAS_DBUTILS:
        try:
            dbutils.widgets.get(name)
        except Exception:
            dbutils.widgets.dropdown(name, default, choices, label or name)
        return dbutils.widgets.get(name)
    return os.getenv(name.upper(), default)


def _as_bool(value):
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


def _split_csv(value):
    return [v.strip() for v in str(value or "").split(",") if v.strip()]


def _find_repo_root():
    cwd = Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "mdp").exists() and (candidate / "notebooks").exists():
            return candidate
    return cwd


REPO_ROOT = _find_repo_root()
print(f"Repo root: {REPO_ROOT}")


In [ ]:
# Configuration
DRY_RUN = _as_bool(_ensure_dropdown_widget("dry_run", "true", ["true", "false"], "Dry run"))
SOURCE_MODE = _ensure_dropdown_widget("source_mode", "auto", ["auto", "refresh", "cache"], "Source mode")
PUBLISH_TO_AGOL = _as_bool(_ensure_dropdown_widget("publish_to_agol", "false", ["false", "true"], "Publish to AGOL"))
WRITE_TO_UNITY_CATALOG = _as_bool(_ensure_dropdown_widget("write_to_unity_catalog", "false", ["false", "true"], "Write Databricks tables"))
PUBLISH_GLOBAL = _as_bool(_ensure_dropdown_widget("publish_global", "true", ["true", "false"], "Include global target"))
SELECTED_REGIONS_RAW = _ensure_text_widget("selected_regions", "all", "Regions: all, none, or comma-separated names")
SELECTED_COUNTRIES_RAW = _ensure_text_widget("selected_countries", "Afghanistan", "Countries: none or comma-separated names")

UC_CATALOG = _ensure_text_widget("uc_catalog", "hive_metastore", "Unity Catalog catalog")
UC_SCHEMA = _ensure_text_widget("uc_schema", "acled", "Unity Catalog schema")
UC_RAW_TABLE = _ensure_text_widget("uc_raw_table", "acled_raw", "Raw ACLED table")
UC_CLEAN_TABLE = _ensure_text_widget("uc_clean_table", "acled_clean", "Clean ACLED table")

METADATA_PATH = Path(_ensure_text_widget("metadata_path", "src/mdp/data/acled_metadata.yml", "ACLED metadata YAML"))
if not METADATA_PATH.is_absolute():
    METADATA_PATH = REPO_ROOT / METADATA_PATH

AGOL_PORTAL_DEFAULT = os.getenv("AGOL_PORTAL_URL", "https://geowb.maps.arcgis.com")
AGOL_PORTAL = _ensure_text_widget("agol_portal", AGOL_PORTAL_DEFAULT, "AGOL portal")
AGOL_SECRET_SCOPE = _ensure_text_widget("agol_secret_scope", "agol-credentials", "AGOL secret scope")
AGOL_USERNAME_KEY = _ensure_text_widget("agol_username_key", "username", "AGOL username secret key")
AGOL_PASSWORD_KEY = _ensure_text_widget("agol_password_key", "password", "AGOL password secret key")
AGOL_TOKEN_EXPIRATION = int(_ensure_text_widget("agol_token_expiration", "120", "AGOL token expiration minutes"))

SOURCE_TABLE = "dbo.acled_country_data"
JDBC_URL = (
    "jdbc:sqlserver://itsoc-lsql-srvr-qa.database.windows.net:1433;"
    "database=fcvrskdboarddb;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "authentication=ActiveDirectoryServicePrincipal"
)
SQL_CLIENT_ID = "ed1c9e19-6040-43d6-8f51-f526920ec894"
SQL_CLIENT_SECRET = dbutils.secrets.get(scope="opsarenascope", key="OPSARENAAPICLIENTSECRET") if HAS_DBUTILS else os.getenv("OPSARENAAPICLIENTSECRET", "")
JDBC_PROPS = {
    "user": SQL_CLIENT_ID,
    "password": SQL_CLIENT_SECRET,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}
if not SQL_CLIENT_SECRET:
    raise RuntimeError("SQL service principal secret is missing. In Databricks, configure opsarenascope/OPSARENAAPICLIENTSECRET.")


def _load_metadata(path):
    if not path.exists():
        raise FileNotFoundError(f"ACLED metadata YAML not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}
    if "targets" not in data or not isinstance(data["targets"], dict):
        raise ValueError(f"Metadata file must define a targets mapping: {path}")
    return data


METADATA = _load_metadata(METADATA_PATH)
DEFAULT_TAGS = METADATA.get("defaults", {}).get("tags", []) or []
BATCH_SIZE = int(METADATA.get("defaults", {}).get("batch_size", 2000))
MAX_RECORD_COUNT = int(METADATA.get("defaults", {}).get("max_record_count", 10000))
LAYER_NAME = METADATA.get("defaults", {}).get("layer_name", "events")

print("Configuration loaded.")
print(f"  dry_run:                 {DRY_RUN}")
print(f"  source_mode:             {SOURCE_MODE}")
print(f"  publish_to_agol:         {PUBLISH_TO_AGOL}")
print(f"  write_to_unity_catalog:  {WRITE_TO_UNITY_CATALOG}")
print(f"  metadata:                {METADATA_PATH}")
print(f"  raw table:               {UC_CATALOG}.{UC_SCHEMA}.{UC_RAW_TABLE}")
print(f"  clean table:             {UC_CATALOG}.{UC_SCHEMA}.{UC_CLEAN_TABLE}")


In [ ]:
# Read ACLED source data, or reuse the raw Delta cache when unchanged
def _quote_ident(value):
    return "`" + str(value).replace("`", "``") + "`"


def _table_identifier(catalog, schema, table):
    return ".".join(_quote_ident(part) for part in [catalog, schema, table])


def _sql_string(value):
    return str(value).replace("'", "''")


RAW_TABLE_FQN = _table_identifier(UC_CATALOG, UC_SCHEMA, UC_RAW_TABLE)
CLEAN_TABLE_FQN = _table_identifier(UC_CATALOG, UC_SCHEMA, UC_CLEAN_TABLE)


def _table_exists(table_fqn):
    try:
        spark.sql(f"DESCRIBE TABLE {table_fqn}").limit(1).collect()
        return True
    except Exception:
        return False


def _table_properties(table_fqn):
    try:
        return {row["key"]: row["value"] for row in spark.sql(f"SHOW TBLPROPERTIES {table_fqn}").collect()}
    except Exception:
        return {}


def _jdbc_scalar_query(query, alias):
    return spark.read.jdbc(JDBC_URL, f"({query}) {alias}", properties=JDBC_PROPS)


def _source_schema_columns():
    schema_df = _jdbc_scalar_query(f"SELECT TOP 0 * FROM {SOURCE_TABLE}", "source_schema_probe")
    return schema_df.columns


def _source_fingerprint(source_columns):
    cols_lower = {c.lower(): c for c in source_columns}
    candidates = [
        "timestamp",
        "updated_at",
        "last_updated",
        "event_date",
        "date",
        "year",
    ]
    max_exprs = []
    used_columns = []
    for candidate in candidates:
        column = cols_lower.get(candidate)
        if column:
            safe_column = "[" + column.replace("]", "]]") + "]"
            alias = "max_" + re.sub(r"[^a-z0-9_]+", "_", candidate.lower())
            max_exprs.append(f"MAX(CONVERT(varchar(64), {safe_column}, 126)) AS {alias}")
            used_columns.append(column)
    select_exprs = ["COUNT_BIG(1) AS row_count", *max_exprs]
    query = f"SELECT {', '.join(select_exprs)} FROM {SOURCE_TABLE}"
    row = _jdbc_scalar_query(query, "source_fingerprint").collect()[0].asDict()
    payload = {
        "source_table": SOURCE_TABLE,
        "row_count": int(row.get("row_count") or 0),
        "fingerprint_columns": used_columns,
        "max_values": {k: str(v) for k, v in row.items() if k != "row_count" and v is not None},
    }
    return json.dumps(payload, sort_keys=True), payload


source_columns = _source_schema_columns()
SOURCE_FINGERPRINT, SOURCE_FINGERPRINT_INFO = _source_fingerprint(source_columns)
raw_table_exists = _table_exists(RAW_TABLE_FQN)
raw_table_properties = _table_properties(RAW_TABLE_FQN) if raw_table_exists else {}
cached_fingerprint = raw_table_properties.get("acled.source_fingerprint")
cache_matches_source = bool(raw_table_exists and cached_fingerprint == SOURCE_FINGERPRINT)

print("Source fingerprint:")
print(f"  source rows:         {SOURCE_FINGERPRINT_INFO['row_count']:,}")
print(f"  fingerprint columns: {SOURCE_FINGERPRINT_INFO['fingerprint_columns']}")
print(f"  raw cache exists:    {raw_table_exists}")
print(f"  cache matches:       {cache_matches_source}")

if SOURCE_MODE == "cache" and not cache_matches_source:
    raise RuntimeError(
        "source_mode=cache requires an existing raw Delta table whose acled.source_fingerprint "
        "matches the SQL source. Use source_mode=auto or source_mode=refresh to rebuild the cache."
    )

USING_CACHED_RAW = SOURCE_MODE in {"auto", "cache"} and cache_matches_source
if USING_CACHED_RAW:
    print(f"Reading raw ACLED data from cached Delta table: {UC_CATALOG}.{UC_SCHEMA}.{UC_RAW_TABLE}")
    df_raw = spark.table(RAW_TABLE_FQN)
else:
    print(f"Reading source table from SQL Server: {SOURCE_TABLE}")
    df_raw = spark.read.jdbc(JDBC_URL, SOURCE_TABLE, properties=JDBC_PROPS)

RAW_ROW_COUNT = df_raw.count()
print(f"Loaded {RAW_ROW_COUNT:,} rows and {len(df_raw.columns)} columns.")
df_raw.printSchema()


In [ ]:
# Unity Catalog helpers
def _quote_ident(value):
    return "`" + str(value).replace("`", "``") + "`"


def _table_identifier(catalog, schema, table):
    return ".".join(_quote_ident(part) for part in [catalog, schema, table])


RAW_TABLE_FQN = _table_identifier(UC_CATALOG, UC_SCHEMA, UC_RAW_TABLE)
CLEAN_TABLE_FQN = _table_identifier(UC_CATALOG, UC_SCHEMA, UC_CLEAN_TABLE)


def _sql_string(value):
    return str(value).replace("'", "''")


def ensure_catalog_schema(catalog, schema):
    if catalog and catalog != "hive_metastore":
        try:
            spark.sql(f"CREATE CATALOG IF NOT EXISTS {_quote_ident(catalog)}")
            print(f"Catalog ready: {catalog}")
        except Exception as exc:
            raise RuntimeError(f"Could not create or access catalog {catalog}: {exc}") from exc
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {_quote_ident(catalog)}.{_quote_ident(schema)}")
    print(f"Schema ready: {catalog}.{schema}")


def add_table_comment(table_fqn, comment):
    try:
        spark.sql(f"COMMENT ON TABLE {table_fqn} IS '{_sql_string(comment)}'")
    except Exception as exc:
        print(f"  Comment skipped for {table_fqn}: {exc}")


def add_column_comments(table_fqn, comments):
    for column, comment in comments.items():
        try:
            spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {_quote_ident(column)} COMMENT '{_sql_string(comment)}'")
        except Exception:
            pass


if WRITE_TO_UNITY_CATALOG and DRY_RUN:
    print("Unity Catalog write requested, but dry_run=true. No Delta tables will be written.")
elif WRITE_TO_UNITY_CATALOG and USING_CACHED_RAW:
    print("Raw table write skipped because source_mode reused the existing raw Delta cache.")
elif WRITE_TO_UNITY_CATALOG:
    ensure_catalog_schema(UC_CATALOG, UC_SCHEMA)
    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(RAW_TABLE_FQN)
    )
    add_table_comment(RAW_TABLE_FQN, "Raw ACLED events ingested from the FCV SQL Server source table.")
    spark.sql(
        f"ALTER TABLE {RAW_TABLE_FQN} SET TBLPROPERTIES ("
        f"'acled.source_fingerprint' = '{_sql_string(SOURCE_FINGERPRINT)}', "
        f"'acled.source_table' = '{_sql_string(SOURCE_TABLE)}', "
        f"'acled.source_mode' = '{_sql_string(SOURCE_MODE)}'"
        f")"
    )
    print(f"Wrote raw Delta table: {UC_CATALOG}.{UC_SCHEMA}.{UC_RAW_TABLE} ({RAW_ROW_COUNT:,} rows)")
else:
    print("Unity Catalog writes disabled. Set write_to_unity_catalog=true and dry_run=false to write Delta tables.")


In [ ]:
# Clean geometry and add World Bank regions
cols_lower = {c.lower(): c for c in df_raw.columns}
lat_col = cols_lower.get("latitude", cols_lower.get("lat"))
lon_col = cols_lower.get("longitude", cols_lower.get("lon", cols_lower.get("long")))
country_col = cols_lower.get("country", cols_lower.get("country_name"))

if not lat_col or not lon_col:
    raise ValueError(f"Cannot find latitude/longitude columns. Available columns: {df_raw.columns}")
if not country_col:
    raise ValueError(f"Cannot find country column. Available columns: {df_raw.columns}")

print(f"Using geometry columns: latitude={lat_col}, longitude={lon_col}")
print(f"Using country column: {country_col}")

df_with_geometry = (
    df_raw
    .withColumn("latitude", F.col(lat_col).cast(DoubleType()))
    .withColumn("longitude", F.col(lon_col).cast(DoubleType()))
)
valid_geometry_filter = (
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull() &
    (F.col("latitude") >= -90) & (F.col("latitude") <= 90) &
    (F.col("longitude") >= -180) & (F.col("longitude") <= 180)
)
valid_geometry_count = df_with_geometry.filter(valid_geometry_filter).count()
invalid_geometry_count = RAW_ROW_COUNT - valid_geometry_count
print(f"Geometry status: valid={valid_geometry_count:,}, invalid_or_missing={invalid_geometry_count:,}")

REGION_MAP = {
    "East Asia & Pacific": [
        "American Samoa", "Australia", "Brunei", "Brunei Darussalam", "Cambodia", "China", "Fiji",
        "French Polynesia", "Guam", "Hong Kong", "Hong Kong SAR China", "Hong Kong SAR, China",
        "Indonesia", "Japan", "Kiribati", "Korea Dem. People's Rep.", "Korea, Dem. People's Rep.",
        "North Korea", "Korea Rep.", "Korea, Rep.", "South Korea", "Lao PDR", "Laos", "Macao",
        "Macao SAR China", "Macao SAR, China", "Malaysia", "Marshall Islands", "Micronesia",
        "Micronesia Fed. Sts.", "Micronesia (Federated States of)", "Mongolia", "Myanmar", "Nauru",
        "New Caledonia", "New Zealand", "Northern Mariana Islands", "Palau", "Papua New Guinea",
        "Philippines", "Samoa", "Singapore", "Solomon Islands", "Taiwan", "Taiwan China", "Thailand",
        "Timor-Leste", "East Timor", "Tonga", "Tuvalu", "Vanuatu", "Vietnam", "Viet Nam",
    ],
    "Europe & Central Asia": [
        "Albania", "Andorra", "Armenia", "Austria", "Azerbaijan", "Belarus", "Belgium",
        "Bosnia and Herzegovina", "Bulgaria", "Channel Islands", "Croatia", "Cyprus", "Czechia",
        "Czech Republic", "Denmark", "Estonia", "Faroe Islands", "Finland", "France", "Georgia",
        "Germany", "Gibraltar", "Greece", "Greenland", "Hungary", "Iceland", "Ireland", "Isle of Man",
        "Italy", "Kazakhstan", "Kosovo", "Kyrgyz Republic", "Kyrgyzstan", "Latvia", "Liechtenstein",
        "Lithuania", "Luxembourg", "Moldova", "Monaco", "Montenegro", "Netherlands", "North Macedonia",
        "Macedonia", "Norway", "Poland", "Portugal", "Romania", "Russia", "Russian Federation",
        "San Marino", "Serbia", "Slovak Republic", "Slovakia", "Slovenia", "Spain", "Sweden",
        "Switzerland", "Tajikistan", "Turkiye", "Turkey", "Turkmenistan", "Ukraine", "United Kingdom",
        "Uzbekistan", "Bailiwick of Guernsey", "Bailiwick of Jersey", "Vatican City",
    ],
    "Latin America & Caribbean": [
        "Anguilla", "Antigua and Barbuda", "Argentina", "Aruba", "Bahamas", "Bahamas, The",
        "Bahamas The", "Barbados", "Belize", "Bolivia", "Brazil", "British Virgin Islands",
        "Caribbean Netherlands", "Cayman Islands", "Chile", "Colombia", "Costa Rica", "Cuba", "Curacao",
        "Dominica", "Dominican Republic", "Ecuador", "El Salvador", "Falkland Islands", "French Guiana",
        "Grenada", "Guadeloupe", "Guatemala", "Guyana", "Haiti", "Honduras", "Jamaica", "Martinique",
        "Mexico", "Montserrat", "Nicaragua", "Panama", "Paraguay", "Peru", "Puerto Rico",
        "Saint-Barthelemy", "Saint-Martin", "Sint Maarten", "Sint Maarten (Dutch part)",
        "Sint Maarten Dutch part", "St. Kitts and Nevis", "Saint Kitts and Nevis", "St. Lucia",
        "Saint Lucia", "St. Martin", "St. Martin (French part)", "St. Martin French part",
        "St. Vincent and the Grenadines", "Saint Vincent and the Grenadines", "Suriname",
        "Trinidad and Tobago", "Turks and Caicos Islands", "Uruguay", "Venezuela", "Venezuela RB",
        "Venezuela, RB", "Virgin Islands", "Virgin Islands (U.S.)", "Virgin Islands U.S.",
        "Virgin Islands, U.S.",
    ],
    "Middle East, North Africa, Afghanistan & Pakistan": [
        "Afghanistan", "Algeria", "Bahrain", "Djibouti", "Egypt", "Egypt Arab Rep.",
        "Egypt, Arab Rep.", "Iran", "Iran Islamic Rep.", "Iran, Islamic Rep.",
        "Iran (Islamic Republic of)", "Iraq", "Israel", "Jordan", "Kuwait", "Lebanon", "Libya",
        "Malta", "Morocco", "Oman", "Pakistan", "Qatar", "Saudi Arabia", "Syria",
        "Syrian Arab Republic", "Tunisia", "United Arab Emirates", "West Bank and Gaza", "Palestine",
        "Yemen", "Yemen Rep.", "Yemen, Rep.",
    ],
    "North America": ["Bermuda", "Canada", "Saint Pierre and Miquelon", "United States", "United States of America", "USA"],
    "South Asia": ["Bangladesh", "Bhutan", "India", "Maldives", "Nepal", "Sri Lanka"],
    "Sub-Saharan Africa": [
        "Angola", "Benin", "Botswana", "British Indian Ocean Territory", "Burkina Faso", "Burundi",
        "Cabo Verde", "Cape Verde", "Cameroon", "Central African Republic", "Chad", "Comoros",
        "Congo Dem. Rep.", "Congo, Dem. Rep.", "Democratic Republic of Congo",
        "Democratic Republic of the Congo", "DRC", "Congo Rep.", "Congo, Rep.", "Republic of Congo",
        "Republic of the Congo", "Congo", "Cote d'Ivoire", "Ivory Coast", "Equatorial Guinea",
        "Eritrea", "Eswatini", "Swaziland", "Ethiopia", "Gabon", "Gambia", "Gambia, The",
        "Gambia The", "Ghana", "Guinea", "Guinea-Bissau", "Kenya", "Lesotho", "Liberia",
        "Madagascar", "Malawi", "Mali", "Mauritania", "Mauritius", "Mayotte", "Mozambique",
        "Namibia", "Niger", "Nigeria", "Reunion", "Rwanda", "Saint Helena, Ascension and Tristan da Cunha",
        "Sao Tome and Principe", "Senegal", "Seychelles", "Sierra Leone", "Somalia", "South Africa",
        "South Sudan", "Sudan", "Tanzania", "Togo", "Uganda", "Zambia", "Zimbabwe",
    ],
}

country_region_rows = [(country.lower(), region) for region, countries in REGION_MAP.items() for country in countries]
region_df = spark.createDataFrame(country_region_rows, ["_country_lower", "wb_region"])

df_clean = (
    df_with_geometry
    .filter(valid_geometry_filter)
    .withColumn("_country_lower", F.lower(F.trim(F.col(country_col))))
    .join(F.broadcast(region_df), "_country_lower", "left")
    .fillna({"wb_region": "Other"})
    .drop("_country_lower")
    .cache()
)
CLEAN_ROW_COUNT = df_clean.count()
unmapped_countries = [
    row[0]
    for row in (
        df_clean.filter(F.col("wb_region") == "Other")
        .select(country_col)
        .distinct()
        .orderBy(country_col)
        .collect()
    )
]
print(f"Cleaned row count: {CLEAN_ROW_COUNT:,}")
if unmapped_countries:
    print(f"Countries not mapped to a World Bank region ({len(unmapped_countries)}): {unmapped_countries}")
else:
    print("All countries mapped to a World Bank region.")


In [ ]:
# Write cleaned Unity Catalog table when enabled
if WRITE_TO_UNITY_CATALOG and DRY_RUN:
    print("Clean table write skipped because dry_run=true.")
elif WRITE_TO_UNITY_CATALOG:
    (
        df_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(CLEAN_TABLE_FQN)
    )
    add_table_comment(CLEAN_TABLE_FQN, "ACLED events with valid WGS84 coordinates and World Bank region assignment.")
    add_column_comments(CLEAN_TABLE_FQN, {
        "latitude": "Latitude in WGS84 decimal degrees.",
        "longitude": "Longitude in WGS84 decimal degrees.",
        "wb_region": "World Bank region assigned from country name using the 2024 classification.",
    })
    print(f"Wrote cleaned Delta table: {UC_CATALOG}.{UC_SCHEMA}.{UC_CLEAN_TABLE} ({CLEAN_ROW_COUNT:,} rows)")
else:
    print("Clean table write disabled.")


In [ ]:
# Build selected publish targets and row-count status
ALL_REGION_NAMES = list(REGION_MAP.keys())
COUNTRY_ISO3_OVERRIDES = {
    "afghanistan": "AFG",
}


def _target_tags(target):
    merged = []
    for tag in [*DEFAULT_TAGS, *(target.get("tags") or [])]:
        tag = str(tag).format(**target).lower()
        if tag not in merged:
            merged.append(tag)
    return merged


def _metadata_targets_by_type(target_type):
    return {key: value for key, value in METADATA["targets"].items() if value.get("type") == target_type}


def _selected_values(raw, all_values):
    values = _split_csv(raw)
    lowered = {v.lower() for v in values}
    if not values or "none" in lowered:
        return []
    if "all" in lowered:
        return list(all_values)
    canonical = {v.lower(): v for v in all_values}
    selected, unknown = [], []
    for value in values:
        match = canonical.get(value.lower())
        if match:
            selected.append(match)
        else:
            unknown.append(value)
    if unknown:
        raise ValueError(f"Unknown selection(s): {unknown}. Allowed values: {all_values}")
    return selected


def _country_selection_values(raw):
    values = _split_csv(raw)
    lowered = {v.lower() for v in values}
    if not values or "none" in lowered:
        return []
    if "all" in lowered:
        return [
            row[0]
            for row in (
                df_clean.select(country_col)
                .where(F.col(country_col).isNotNull())
                .distinct()
                .orderBy(country_col)
                .collect()
            )
        ]
    return values


def _country_name_and_iso3(selection):
    selection = selection.strip()
    if ":" in selection:
        left, right = [part.strip() for part in selection.split(":", 1)]
        if len(left) == 3:
            return right, left.upper()
        if len(right) == 3:
            return left, right.upper()
    if "|" in selection:
        left, right = [part.strip() for part in selection.split("|", 1)]
        if len(left) == 3:
            return right, left.upper()
        if len(right) == 3:
            return left, right.upper()
    country_name = selection
    override = COUNTRY_ISO3_OVERRIDES.get(country_name.lower())
    if override:
        return country_name, override
    try:
        import pycountry
        match = pycountry.countries.lookup(country_name)
        return country_name, match.alpha_3.upper()
    except Exception:
        raise ValueError(
            f"Could not infer ISO3 for selected country '{selection}'. "
            "Use 'ISO3:Country Name', for example 'AFG:Afghanistan'."
        )


country_names_in_data = {
    row[0].lower(): row[0]
    for row in (
        df_clean.select(country_col)
        .where(F.col(country_col).isNotNull())
        .distinct()
        .collect()
    )
}


def _generic_country_target(country_name, iso3):
    template = (METADATA.get("templates") or {}).get("country") or {}
    values = {
        "country_name": country_name,
        "iso3": iso3.lower(),
        "iso3_upper": iso3.upper(),
    }
    return {
        "type": "country",
        "selector": country_name,
        "country_name": country_name,
        "iso3": iso3.lower(),
        "item_id": template.get("item_id"),
        "title": template.get("title_template", "ACLED Conflict Events - {country_name}").format(**values),
        "snippet": template.get("snippet_template", "Armed conflict and protest event data from ACLED for {country_name}.").format(**values),
        "tags": [str(tag).format(**values).lower() for tag in template.get("tags", ["{iso3}"])],
    }


def _subset_for_target(target):
    target_type = target.get("type")
    selector = target.get("selector")
    if target_type == "global":
        return df_clean
    if target_type == "region":
        return df_clean.filter(F.col("wb_region") == selector)
    if target_type == "country":
        return df_clean.filter(F.col(country_col) == selector)
    raise ValueError(f"Unsupported target type: {target_type}")


selected_regions = _selected_values(SELECTED_REGIONS_RAW, ALL_REGION_NAMES)
explicit_country_targets = _metadata_targets_by_type("country")
explicit_country_by_name = {
    target.get("selector", "").lower(): (key, target)
    for key, target in explicit_country_targets.items()
}
selected_country_entries = _country_selection_values(SELECTED_COUNTRIES_RAW)

selected_targets = []
for key, target in METADATA["targets"].items():
    target_type = target.get("type")
    selector = target.get("selector")
    if target_type == "global" and PUBLISH_GLOBAL:
        selected_targets.append((key, target))
    elif target_type == "region" and selector in selected_regions:
        selected_targets.append((key, target))

for entry in selected_country_entries:
    country_name, iso3 = _country_name_and_iso3(entry)
    canonical_country_name = country_names_in_data.get(country_name.lower(), country_name)
    explicit = explicit_country_by_name.get(canonical_country_name.lower())
    if explicit:
        selected_targets.append(explicit)
    else:
        selected_targets.append((f"country_{iso3.lower()}", _generic_country_target(canonical_country_name, iso3)))

if not selected_targets:
    raise ValueError("No publish targets selected. Enable global, select at least one region, or select at least one country.")

seen_keys = set()
deduped_targets = []
for key, target in selected_targets:
    if key in seen_keys:
        continue
    seen_keys.add(key)
    deduped_targets.append((key, target))

TARGET_STATUS = []
print("Selected target status:")
for key, target in deduped_targets:
    subset = _subset_for_target(target)
    row_count = subset.count()
    TARGET_STATUS.append({"key": key, "target": target, "row_count": row_count})
    item_id = target.get("item_id") or "(new item)"
    print(f"  {key}: type={target.get('type')}, selector={target.get('selector')}, rows={row_count:,}, item_id={item_id}")

zero_targets = [status["key"] for status in TARGET_STATUS if status["row_count"] == 0]
if zero_targets:
    print(f"Targets with zero rows will be skipped: {zero_targets}")


In [ ]:
# AGOL username/password authentication and status checks
PLACEHOLDER_VALUES = {"", "your_username", "your_password", "PASTE_HERE"}


def _get_secret(scope, key, env_name=None):
    if HAS_DBUTILS:
        try:
            value = dbutils.secrets.get(scope=scope, key=key)
            if value:
                return value
        except Exception as exc:
            print(f"  Secret lookup skipped for {scope}/{key}: {exc}")
    return os.getenv(env_name or key.upper(), "")


def _configured_secret(value):
    return bool(value) and value not in PLACEHOLDER_VALUES and not value.startswith("PASTE_")


def _raise_agol_error(context, data):
    if "error" not in data:
        return
    err = data["error"]
    if isinstance(err, dict):
        msg = err.get("message") or err.get("error_description") or err
        details = err.get("details") or []
        raise RuntimeError(f"{context}: {msg}; details={details}; raw={err}")
    raise RuntimeError(f"{context}: {err}")


def agol_request(method, url, *, context, token=None, params=None, data=None, timeout=30):
    params = dict(params or {})
    data = dict(data or {})
    if token:
        if method.upper() == "GET":
            params["token"] = token
        else:
            data["token"] = token
    if method.upper() == "GET":
        response = requests.get(url, params=params, timeout=timeout)
    else:
        response = requests.post(url, data=data, timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    _raise_agol_error(context, payload)
    return payload


def get_agol_user_token(portal, username, password, expiration=120):
    data = agol_request(
        "POST",
        f"{portal}/sharing/rest/generateToken",
        context="AGOL username/password login failed",
        data={
            "username": username,
            "password": password,
            "client": "referer",
            "referer": portal,
            "expiration": expiration,
            "f": "json",
        },
    )
    token = data.get("token")
    if not token:
        raise RuntimeError(f"AGOL login returned no token: {data}")
    return token


def validate_agol_user(portal, token):
    self_info = agol_request("GET", f"{portal}/sharing/rest/community/self", context="AGOL /community/self failed", token=token, params={"f": "json"})
    username = self_info.get("username")
    if not username:
        raise RuntimeError(f"Expected a user token, but AGOL did not return a username: {self_info}")
    content = agol_request("GET", f"{portal}/sharing/rest/content/users/{username}", context=f"Cannot access AGOL content for {username}", token=token, params={"f": "json", "num": 1})
    dry = requests.post(
        f"{portal}/sharing/rest/content/users/{username}/createService",
        data={"f": "json", "token": token, "createParameters": "{}", "outputType": "featureService"},
        timeout=30,
    )
    dry.raise_for_status()
    dry_payload = dry.json()
    if dry_payload.get("error", {}).get("code") == 403:
        raise RuntimeError(f"AGOL user {username} cannot create hosted feature services: {dry_payload['error']}")
    return {
        "username": username,
        "full_name": self_info.get("fullName"),
        "role": self_info.get("role"),
        "privileges": self_info.get("privileges") or [],
        "folders": len(content.get("folders", []) or []),
        "create_service_dry_run": dry_payload.get("error", {}).get("code", "allowed past auth"),
    }


AGOL_TOKEN = None
AGOL_IDENTITY = None
AGOL_USERNAME = ""

if PUBLISH_TO_AGOL:
    AGOL_USERNAME = _get_secret(AGOL_SECRET_SCOPE, AGOL_USERNAME_KEY, "AGOL_USERNAME")
    AGOL_PASSWORD = _get_secret(AGOL_SECRET_SCOPE, AGOL_PASSWORD_KEY, "AGOL_PASSWORD")
    if not _configured_secret(AGOL_USERNAME) or not _configured_secret(AGOL_PASSWORD):
        raise RuntimeError(
            f"AGOL credentials are missing. Configure Databricks secrets "
            f"{AGOL_SECRET_SCOPE}/{AGOL_USERNAME_KEY} and {AGOL_SECRET_SCOPE}/{AGOL_PASSWORD_KEY}."
        )
    AGOL_TOKEN = get_agol_user_token(AGOL_PORTAL, AGOL_USERNAME, AGOL_PASSWORD, AGOL_TOKEN_EXPIRATION)
    AGOL_IDENTITY = validate_agol_user(AGOL_PORTAL, AGOL_TOKEN)
    print("AGOL user token validated.")
    print(f"  User:          {AGOL_IDENTITY['username']}")
    print(f"  Name:          {AGOL_IDENTITY['full_name']}")
    print(f"  Role:          {AGOL_IDENTITY['role']}")
    print(f"  Folders:       {AGOL_IDENTITY['folders']}")
    print(f"  createService: passed auth check (dry-run code={AGOL_IDENTITY['create_service_dry_run']})")
else:
    print("AGOL publishing disabled. Set publish_to_agol=true to validate AGOL credentials and item permissions.")


In [ ]:
# AGOL publishing helpers
def _spark_type_to_esri(spark_type):
    mapping = {
        StringType: "esriFieldTypeString",
        IntegerType: "esriFieldTypeInteger",
        LongType: "esriFieldTypeInteger",
        ShortType: "esriFieldTypeSmallInteger",
        ByteType: "esriFieldTypeSmallInteger",
        FloatType: "esriFieldTypeDouble",
        DoubleType: "esriFieldTypeDouble",
        DecimalType: "esriFieldTypeDouble",
        DateType: "esriFieldTypeDate",
        TimestampType: "esriFieldTypeDate",
        BooleanType: "esriFieldTypeSmallInteger",
    }
    return mapping.get(type(spark_type), "esriFieldTypeString")


def _service_name(title):
    name = title.replace("&", "and")
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name).strip("_")
    return name[:90] or "acled_events"


def _json_default(value):
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)


def _feature_batches(df, lat_col="latitude", lon_col="longitude", batch_size=BATCH_SIZE):
    batch = []
    for row in df.toLocalIterator():
        attrs = row.asDict(recursive=True)
        x = attrs.pop(lon_col, None)
        y = attrs.pop(lat_col, None)
        for key, value in list(attrs.items()):
            if hasattr(value, "isoformat"):
                attrs[key] = value.isoformat()
        batch.append({"geometry": {"x": x, "y": y, "spatialReference": {"wkid": 4326}}, "attributes": attrs})
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def _target_description(target):
    dataset = METADATA.get("dataset", {})
    template = target.get("description_template") or dataset.get("description_template", "{coverage}")
    contacts = "; ".join(dataset.get("contacts") or [])
    return template.format(
        source_table=dataset.get("source_table", SOURCE_TABLE),
        coverage=target.get("title") or target.get("selector"),
        country_name=target.get("country_name") or target.get("selector") or "",
        iso3=target.get("iso3", ""),
        iso3_upper=str(target.get("iso3", "")).upper(),
        update_date=date.today().isoformat(),
        last_updated=dataset.get("last_updated", date.today().isoformat()),
        contacts=contacts,
    )


def _target_metadata(target):
    return {
        "title": target["title"],
        "snippet": target.get("snippet", ""),
        "description": _target_description(target),
        "tags": ",".join(_target_tags(target)),
        "licenseInfo": METADATA.get("dataset", {}).get("license_info", ""),
    }


def get_item(portal, token, item_id):
    return agol_request("GET", f"{portal}/sharing/rest/content/items/{item_id}", context=f"Item lookup failed for {item_id}", token=token, params={"f": "json"})


def find_feature_service_by_title(portal, token, username, title):
    data = agol_request(
        "GET",
        f"{portal}/sharing/rest/search",
        context=f"AGOL search failed for {title}",
        token=token,
        params={"f": "json", "q": f'title:"{title}" AND owner:{username} AND type:"Feature Service"', "num": 10},
    )
    matches = [item for item in data.get("results", []) if (item.get("title") or "").strip() == title]
    if not matches:
        return None
    return sorted(matches, key=lambda item: item.get("modified") or 0, reverse=True)[0]


def validate_item_permission(portal, token, username, identity, item_id):
    item = get_item(portal, token, item_id)
    owner = (item.get("owner") or "").lower()
    is_owner = owner == username.lower()
    is_admin = identity and identity.get("role") == "org_admin"
    if not is_owner and not is_admin:
        raise RuntimeError(f"AGOL item {item_id} is owned by '{item.get('owner')}', but authenticated user is '{username}' and is not org_admin.")
    if item.get("type") != "Feature Service":
        raise RuntimeError(f"AGOL item {item_id} is type '{item.get('type')}', expected Feature Service.")
    return item


def get_feature_service_url(portal, token, item_id):
    item = get_item(portal, token, item_id)
    if not item.get("url"):
        raise RuntimeError(f"AGOL item {item_id} has no service URL.")
    return item["url"]


def service_info(service_url, token):
    return agol_request("GET", service_url, context=f"Could not read service info: {service_url}", token=token, params={"f": "json"})


def create_feature_service(portal, username, token, target):
    title = target["title"]
    create_params = json.dumps({
        "name": _service_name(title),
        "serviceDescription": target.get("snippet", ""),
        "hasStaticData": False,
        "maxRecordCount": MAX_RECORD_COUNT,
        "supportedQueryFormats": "JSON",
        "capabilities": "Query,Editing,Create,Update,Delete,Extract",
        "spatialReference": {"wkid": 4326},
        "allowGeometryUpdates": True,
        "units": "esriDecimalDegrees",
        "xssPreventionInfo": {"xssPreventionEnabled": True, "xssPreventionRule": "InputOnly", "xssInputRule": "rejectInvalid"},
    })
    result = agol_request(
        "POST",
        f"{portal}/sharing/rest/content/users/{username}/createService",
        context=f"createService failed for {title}",
        token=token,
        data={"f": "json", "createParameters": create_params, "outputType": "featureService", **_target_metadata(target)},
        timeout=60,
    )
    if not result.get("success"):
        raise RuntimeError(f"createService did not succeed for {title}: {result}")
    return result["itemId"], result["serviceurl"]


def add_layer_to_service(service_url, token, df, layer_name=LAYER_NAME):
    fields = []
    for field in df.schema.fields:
        if field.name in ("latitude", "longitude"):
            continue
        esri_type = _spark_type_to_esri(field.dataType)
        entry = {"name": field.name, "type": esri_type, "alias": field.name, "nullable": True}
        if esri_type == "esriFieldTypeString":
            entry["length"] = 1024
        fields.append(entry)
    layer_def = json.dumps({"layers": [{"name": layer_name, "type": "Feature Layer", "geometryType": "esriGeometryPoint", "fields": fields, "capabilities": "Query,Editing,Create,Update,Delete,Extract"}]})
    result = agol_request("POST", f"{service_url}/addToDefinition", context=f"addToDefinition failed for {service_url}", token=token, data={"f": "json", "addToDefinition": layer_def}, timeout=60)
    if not result.get("success"):
        raise RuntimeError(f"addToDefinition did not succeed: {result}")


def _admin_service_url(service_url):
    if "/rest/services/" not in service_url:
        raise RuntimeError(f"Cannot derive admin URL from service URL: {service_url}")
    return service_url.replace("/rest/services/", "/rest/admin/services/")


def _capability_set(raw):
    return {cap.strip() for cap in str(raw or "").split(",") if cap.strip()}


def _capability_string(caps):
    preferred = ["Query", "Create", "Update", "Delete", "Editing", "Extract"]
    ordered = [cap for cap in preferred if cap in caps]
    ordered.extend(sorted(cap for cap in caps if cap not in preferred))
    return ",".join(ordered)


def ensure_service_capabilities(service_url, token, layer_index=None):
    """Ensure update/download capabilities are enabled without removing existing capabilities."""
    info = service_info(service_url, token)
    current = _capability_set(info.get("capabilities", ""))
    needed = {"Query", "Create", "Update", "Delete", "Editing", "Extract"}
    admin_url = _admin_service_url(service_url)

    if not needed.issubset(current):
        new_caps = _capability_string(current | needed)
        result = agol_request(
            "POST",
            f"{admin_url}/updateDefinition",
            context=f"Could not update service capabilities for {service_url}",
            token=token,
            data={"f": "json", "updateDefinition": json.dumps({"capabilities": new_caps})},
            timeout=60,
        )
        if not result.get("success"):
            raise RuntimeError(f"Could not update service capabilities for {service_url}: {result}")
        print(f"  Service capabilities: {new_caps}")

    if layer_index is None:
        return

    layer_info = agol_request(
        "GET",
        f"{service_url}/{layer_index}",
        context=f"Could not read layer capabilities for {service_url}/{layer_index}",
        token=token,
        params={"f": "json"},
    )
    layer_current = _capability_set(layer_info.get("capabilities", ""))
    if needed.issubset(layer_current):
        return
    layer_caps = _capability_string(layer_current | needed)
    result = agol_request(
        "POST",
        f"{admin_url}/{layer_index}/updateDefinition",
        context=f"Could not update layer capabilities for {service_url}/{layer_index}",
        token=token,
        data={"f": "json", "updateDefinition": json.dumps({"capabilities": layer_caps})},
        timeout=60,
    )
    if not result.get("success"):
        raise RuntimeError(f"Could not update layer capabilities for {service_url}/{layer_index}: {result}")
    print(f"  Layer {layer_index} capabilities: {layer_caps}")


def truncate_layer(service_url, layer_index, token):
    """Clear a hosted feature layer using the admin truncate operation."""
    admin_url = _admin_service_url(service_url)
    result = agol_request(
        "POST",
        f"{admin_url}/{layer_index}/truncate",
        context=f"truncate failed for {service_url}/{layer_index}",
        token=token,
        data={"f": "json", "attachmentOnly": "false", "async": "false"},
        timeout=300,
    )
    if not result.get("success", True):
        raise RuntimeError(f"truncate did not succeed for {service_url}/{layer_index}: {result}")
    print(f"  Truncated existing features from layer {layer_index}.")


def _chunks(values, size):
    for start in range(0, len(values), size):
        yield values[start:start + size]


def delete_layer_features_by_object_id(service_url, layer_index, token, batch_size=5000):
    """Fallback clear path for services where truncate is unavailable."""
    id_payload = agol_request(
        "GET",
        f"{service_url}/{layer_index}/query",
        context=f"Could not query object IDs for {service_url}/{layer_index}",
        token=token,
        params={"f": "json", "where": "1=1", "returnIdsOnly": "true"},
        timeout=120,
    )
    object_ids = id_payload.get("objectIds") or []
    if not object_ids:
        print(f"  Layer {layer_index} already empty.")
        return 0

    deleted = 0
    for batch_number, object_id_batch in enumerate(_chunks(object_ids, batch_size), start=1):
        result = agol_request(
            "POST",
            f"{service_url}/{layer_index}/deleteFeatures",
            context=f"deleteFeatures objectId batch {batch_number} failed for {service_url}/{layer_index}",
            token=token,
            data={"f": "json", "objectIds": ",".join(str(oid) for oid in object_id_batch)},
            timeout=300,
        )
        delete_results = result.get("deleteResults") or []
        failed = [r for r in delete_results if not r.get("success")]
        if failed:
            raise RuntimeError(f"deleteFeatures batch {batch_number} had failed deletes: {failed[:3]}")
        deleted += len(delete_results) or len(object_id_batch)
        print(f"  Delete batch {batch_number}: deleted {len(delete_results) or len(object_id_batch):,} features")
    return deleted


def clear_layer(service_url, layer_index, token):
    try:
        truncate_layer(service_url, layer_index, token)
        return
    except Exception as exc:
        print(f"  Truncate unavailable or failed; falling back to batched delete by objectId: {exc}")
    deleted = delete_layer_features_by_object_id(service_url, layer_index, token)
    print(f"  Cleared existing features from layer {layer_index}; deleted={deleted:,}")


def append_spark_features(service_url, layer_index, token, df):
    total_added = 0
    for batch_number, batch in enumerate(_feature_batches(df), start=1):
        result = agol_request("POST", f"{service_url}/{layer_index}/applyEdits", context=f"applyEdits failed for batch {batch_number}", token=token, data={"f": "json", "adds": json.dumps(batch, default=_json_default)}, timeout=300)
        add_results = result.get("addResults") or []
        failed = [r for r in add_results if not r.get("success")]
        if failed:
            raise RuntimeError(f"applyEdits batch {batch_number} had failed adds: {failed[:3]}")
        added = sum(1 for r in add_results if r.get("success"))
        total_added += added
        print(f"  Batch {batch_number}: added {added:,} features")
    return total_added


def update_item_metadata(portal, username, token, item_id, target):
    result = agol_request("POST", f"{portal}/sharing/rest/content/users/{username}/items/{item_id}/update", context=f"Metadata update failed for {item_id}", token=token, data={"f": "json", **_target_metadata(target)}, timeout=60)
    if not result.get("success"):
        raise RuntimeError(f"Metadata update did not succeed for {item_id}: {result}")


def publish_target(portal, username, token, identity, target, subset):
    item_id = target.get("item_id")
    service_url = None
    created = False
    if item_id:
        validate_item_permission(portal, token, username, identity, item_id)
        service_url = get_feature_service_url(portal, token, item_id)
    else:
        existing = find_feature_service_by_title(portal, token, username, target["title"])
        if existing:
            item_id = existing["id"]
            validate_item_permission(portal, token, username, identity, item_id)
            service_url = existing.get("url") or get_feature_service_url(portal, token, item_id)
            print(f"  Reusing existing item with matching title: {item_id}")
        else:
            item_id, service_url = create_feature_service(portal, username, token, target)
            created = True
            print(f"  Created item_id={item_id}. Add this to acled_metadata.yml for target {target.get('selector')}.")
    info = service_info(service_url, token)
    layers = info.get("layers") or []
    if created or not layers:
        add_layer_to_service(service_url, token, subset)
        layers = service_info(service_url, token).get("layers") or []
    if not layers:
        raise RuntimeError(f"Feature service has no layers after addToDefinition: {service_url}")
    layer_index = layers[0].get("id", 0)
    ensure_service_capabilities(service_url, token, layer_index)
    clear_layer(service_url, layer_index, token)
    added = append_spark_features(service_url, layer_index, token, subset)
    update_item_metadata(portal, username, token, item_id, target)
    return item_id, added


print("AGOL publish helpers loaded.")


In [ ]:
# Publish or dry-run selected targets
if not PUBLISH_TO_AGOL:
    print("AGOL publishing skipped because publish_to_agol=false.")
elif DRY_RUN:
    print("Dry run: validating selected AGOL items without creating, clearing, appending, or updating metadata.")
    for status in TARGET_STATUS:
        target = status["target"]
        item_id = target.get("item_id")
        print(f"\n=== {status['key']} ===")
        print(f"  selector: {target.get('selector')}")
        print(f"  rows:     {status['row_count']:,}")
        if item_id:
            item = validate_item_permission(AGOL_PORTAL, AGOL_TOKEN, AGOL_IDENTITY["username"], AGOL_IDENTITY, item_id)
            print(f"  item:     OK ({item_id}, owner={item.get('owner')})")
        else:
            existing = find_feature_service_by_title(AGOL_PORTAL, AGOL_TOKEN, AGOL_IDENTITY["username"], target["title"])
            if existing:
                print(f"  item:     metadata item_id is blank, but matching AGOL item exists: {existing['id']}")
            else:
                print("  item:     metadata item_id is blank; publish run will create a new hosted feature service")
else:
    published = []
    for status in TARGET_STATUS:
        target = status["target"]
        if status["row_count"] == 0:
            print(f"\n=== {status['key']} ===")
            print("  Skipped because selected subset has zero rows.")
            continue
        print(f"\n=== {status['key']} ===")
        subset = _subset_for_target(target)
        item_id, added = publish_target(AGOL_PORTAL, AGOL_IDENTITY["username"], AGOL_TOKEN, AGOL_IDENTITY, target, subset)
        published.append({"key": status["key"], "item_id": item_id, "features": added})
        print(f"  Published item_id={item_id}, features={added:,}")
    print("\nPublishing complete.")
    for item in published:
        print(f"  {item['key']}: {item['item_id']} ({item['features']:,} features)")


In [ ]:
# Run summary
print("ACLED pipeline completed.")
print(f"  Source rows:     {RAW_ROW_COUNT:,}")
print(f"  Source mode:     {SOURCE_MODE}")
print(f"  Cached raw:      {USING_CACHED_RAW}")
print(f"  Clean rows:      {CLEAN_ROW_COUNT:,}")
print(f"  Dry run:         {DRY_RUN}")
print(f"  UC write:        {WRITE_TO_UNITY_CATALOG and not DRY_RUN}")
print(f"  AGOL publish:    {PUBLISH_TO_AGOL and not DRY_RUN}")
print(f"  Targets checked: {len(TARGET_STATUS)}")
